# Visual Inspection: Remote Sensing Indicators by City

Interactive maps overlaying sampled business coordinates on the underlying GEE raster layers used to derive each indicator. Set the `CITY` parameter in the cell below to switch between study areas.

In [ ]:
# ============================================================
# SETUP: Set city parameter, load data, initialise GEE
# ============================================================

CITY = "Lagos"  # Must be a city present in all_indicators.csv (listed below)

import ee
import geemap
import pandas as pd
from pathlib import Path

ee.Initialize(project="ee-geogrids")

# Load indicator results
root = Path.cwd() if Path("data/output/all_indicators.csv").exists() else Path.cwd().parent
all_df = pd.read_csv(root / "data/output/all_indicators.csv")

# A city is absent whenever it was excluded from the extraction run. Without
# this guard the filter yields 0 rows, the centre coordinates become NaN, and
# the next cell fails deep inside GEE with an opaque
# 'Invalid JSON payload ... {"constantValue": [NaN, NaN]}' error.
available = sorted(all_df["city"].unique())
if CITY not in available:
    raise ValueError(
        f"No rows for CITY={CITY!r} in all_indicators.csv.\n"
        f"  Cities present: {available}\n"
        f"  A city is missing when its country was excluded from the run via "
        f"gsmm.include_countries in config.yaml.\n"
        f"  To add it:  edit that key, then "
        f"`cd python && python3 prepare_gsmm_input.py && python3 run_all.py`"
    )

df = all_df[all_df["city"] == CITY].copy()
print(f"City: {CITY} — {len(df)} businesses")
print(f"Cities available in this extract: {available}")

# Centre coordinates for map
center_lat = df["latitude"].mean()
center_lon = df["longitude"].mean()

# Analysis windows, read from the data itself rather than recomputed. The
# pipeline uses ONE fixed-length window per city (config: time_window), not a
# per-fieldwork-date window, so deriving it from fieldwork_date here would show
# a raster that disagrees with the extracted point values.
HEAT_START, HEAT_END = df["heat_window_start"].iloc[0], df["heat_window_end"].iloc[0]
RAIN_START, RAIN_END = df["rain_window_start"].iloc[0], df["rain_window_end"].iloc[0]
AOD_START,  AOD_END  = df["aod_window_start"].iloc[0],  df["aod_window_end"].iloc[0]
NTL_END   = HEAT_END
NTL_START = (pd.Timestamp(NTL_END) - pd.DateOffset(months=12)).strftime("%Y-%m-%d")

# Some indicators are legitimately missing at a few points — e.g. 24 Lagos
# businesses sit in a permanently masked MODIS pixel (see data_dictionary.md).
# GEE rejects NaN in a Feature property, so pass None and let the styling cells
# skip those points rather than crash, and never substitute 0 (which would read
# as a real "zero hot days" value).
def prop(v, nd=None, as_int=False):
    if pd.isna(v):
        return None
    return int(v) if as_int else (round(float(v), nd) if nd is not None else float(v))

features = []
for _, row in df.iterrows():
    if pd.isna(row["longitude"]) or pd.isna(row["latitude"]):
        continue
    geom = ee.Geometry.Point([row["longitude"], row["latitude"]])
    features.append(ee.Feature(geom, {
        "business_id": row["business_id"],
        "elevation_m": prop(row["elevation_m"], as_int=True),
        "heat_days_gt40c": prop(row["heat_days_gt40c"], as_int=True),
        "lst_mean_c": prop(row["lst_mean_c"], 1),
        "hand_m": prop(row["hand_m"], 1),
        "hand_flood_vulnerable": prop(row["hand_flood_vulnerable"], as_int=True),
        "canopy_fraction_50m": prop(row["canopy_fraction_50m"], 3),
        "canopy_fraction_150m": prop(row["canopy_fraction_150m"], 3),
        "rain_days_gt20mm": prop(row["rain_days_gt20mm"], as_int=True),
        "aod_mean": prop(row["aod_mean"], 3),
        "ntl_mean_radiance": prop(row["ntl_mean_radiance"], 2),
        "builtup_fraction_150m": prop(row["builtup_fraction_150m"], 3),
    }))
fc = ee.FeatureCollection(features)

n_nodata = int(df["heat_days_gt40c"].isna().sum())
if n_nodata:
    print(f"Note: {n_nodata} point(s) have no MODIS LST data (permanently masked "
          f"pixel); they are drawn in grey on the heat map, not as zero.")

print(f"Centre: {center_lat:.4f}, {center_lon:.4f}")
print(f"Heat/rain/AOD window: {HEAT_START} to {HEAT_END}")
print("Ready.")

---
## Indicator 1: Elevation (SRTM 30m)

In [7]:
# ============================================================
# INDICATOR 1: Elevation — SRTM 30m
# ============================================================

srtm = ee.Image("USGS/SRTMGL1_003").select("elevation")

# Compute local min/max for colour stretch
bbox = ee.Geometry.Point([center_lon, center_lat]).buffer(5000)
stats = srtm.reduceRegion(ee.Reducer.minMax(), bbox, 30).getInfo()
elev_min = stats.get("elevation_min", 0)
elev_max = stats.get("elevation_max", 500)

elev_palette = ["#2166ac", "#67a9cf", "#d1e5f0", "#fddbc7", "#ef8a62", "#b2182b"]

m1 = geemap.Map(center=[center_lat, center_lon], zoom=14)
m1.add_basemap("CartoDB.Positron")

m1.addLayer(
    srtm,
    {"min": elev_min, "max": elev_max, "palette": elev_palette},
    "SRTM Elevation (m)",
)

# Style points: colour by extracted elevation
styled_points = fc.map(
    lambda f: f.set("style", {
        "color": "#000000",
        "pointSize": 6,
        "fillColor": "#FFFF00",
        "width": 1.5,
    })
)
m1.addLayer(styled_points.style(**{"styleProperty": "style"}), {}, "Business locations")

m1.add_colorbar(
    vis_params={"min": elev_min, "max": elev_max, "palette": elev_palette},
    label="Elevation (m above sea level)",
)

m1

EEException: Invalid JSON payload received. Unexpected token.
{"constantValue": [NaN, NaN]}}}}}}}, "i
                   ^

---
## Indicator 2: Extreme Heat Days (MODIS Daytime LST)

Shows the mean daytime Land Surface Temperature over the 2-year trailing window. Business points are coloured by their extracted `heat_days_gt40c` value.

In [ ]:
# ============================================================
# INDICATOR 2: Heat — MODIS LST mean + max over trailing 2-year window
# ============================================================

# Use the most common fieldwork date window for this city's raster display
# Window used by the extraction, read from the data (not recomputed)
start_date, end_date = HEAT_START, HEAT_END

modis = (
    ee.ImageCollection("MODIS/061/MOD11A1")
    .filterDate(start_date, end_date)
    .select("LST_Day_1km")
)

# Convert to Celsius
modis_c = modis.map(lambda img: img.multiply(0.02).add(-273.15))
lst_mean = modis_c.mean()
lst_max = modis_c.max()

lst_palette = ["#313695", "#4575b4", "#74add1", "#abd9e9", "#fee090", "#fdae61", "#f46d43", "#d73027", "#a50026"]

m2 = geemap.Map(center=[center_lat, center_lon], zoom=13)
m2.add_basemap("CartoDB.DarkMatter")

m2.addLayer(
    lst_mean,
    {"min": 20, "max": 50, "palette": lst_palette},
    f"Mean Daytime LST (°C) [{start_date} to {end_date}]",
)

m2.addLayer(
    lst_max,
    {"min": 30, "max": 65, "palette": lst_palette},
    f"Max Daytime LST (°C) [{start_date} to {end_date}]",
    shown=False,
)

# Colour points by heat_days_gt40c: green (0 days) → red (many days).
# Points with NO MODIS data are split into a separate grey layer — colouring
# them as 0 would misread "pixel never observed" as "never exceeded 40C".
heat_max = int(max(df["heat_days_gt40c"].max(), 1))
fc_heat = fc.filter(ee.Filter.notNull(["heat_days_gt40c"]))
fc_nodata = fc.filter(ee.Filter.eq("heat_days_gt40c", None))

def style_heat(f):
    days = ee.Number(f.get("heat_days_gt40c"))
    ratio = days.divide(heat_max).min(1)
    r = ratio.multiply(255).toInt()
    g = ee.Number(1).subtract(ratio).multiply(255).toInt()
    color = ee.String("#").cat(
        r.format("%02x")
    ).cat(
        g.format("%02x")
    ).cat("00")
    return f.set("style", {
        "color": "#FFFFFF",
        "pointSize": 7,
        "fillColor": color,
        "width": 1,
    })

m2.addLayer(
    fc_heat.map(style_heat).style(**{"styleProperty": "style"}),
    {},
    f"Businesses (colour = days > 40°C, max={heat_max})",
)

if fc_nodata.size().getInfo():
    m2.addLayer(
        fc_nodata.map(lambda f: f.set("style", {
            "color": "#FFFFFF", "pointSize": 7, "fillColor": "#888888", "width": 1,
        })).style(**{"styleProperty": "style"}),
        {},
        "Businesses (no MODIS LST data)",
    )

m2.add_colorbar(
    vis_params={"min": 20, "max": 50, "palette": lst_palette},
    label="Mean Daytime LST (°C)",
)

m2

---
## Indicator 3: Flood Vulnerability (HAND + JRC Surface Water)

Shows MERIT Hydro HAND (Height Above Nearest Drainage) with JRC maximum water extent overlay. Business points are coloured by flood vulnerability classification.

In [13]:
# ============================================================
# INDICATOR 3: Flood — HAND (MERIT Hydro) + JRC Surface Water
# ============================================================

hand = ee.Image("MERIT/Hydro/v1_0_1").select("hnd")
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
jrc_max_extent = jrc.select("max_extent")
jrc_recurrence = jrc.select("recurrence")

hand_palette = ["#08306b", "#2171b5", "#6baed6", "#bdd7e7", "#f7e4c1", "#d9a441", "#8c510a"]

m3 = geemap.Map(center=[center_lat, center_lon], zoom=14)
m3.add_basemap("CartoDB.Positron")

# HAND layer: blue (low, flood-prone) → brown (high, safe)
m3.addLayer(
    hand.updateMask(hand.lte(50)),
    {"min": 0, "max": 30, "palette": hand_palette},
    "HAND — Height Above Nearest Drainage (m)",
)

# JRC max water extent as semi-transparent blue overlay
m3.addLayer(
    jrc_max_extent.updateMask(jrc_max_extent.eq(1)),
    {"palette": ["#0000FF"], "opacity": 0.35},
    "JRC Max Water Extent (1984–2021)",
)

# JRC recurrence
m3.addLayer(
    jrc_recurrence.updateMask(jrc_recurrence.gt(0)),
    {"min": 0, "max": 100, "palette": ["#c6dbef", "#4292c6", "#08519c"], "opacity": 0.5},
    "JRC Water Recurrence (%)",
    shown=False,
)

# Points: red = flood vulnerable (HAND ≤ 5m), green = not
def style_flood(f):
    vuln = ee.Number(f.get("hand_flood_vulnerable"))
    color = ee.Algorithms.If(vuln.eq(1), "#d62728", "#2ca02c")
    return f.set("style", {
        "color": "#000000",
        "pointSize": 7,
        "fillColor": color,
        "width": 1.5,
    })

m3.addLayer(
    fc.map(style_flood).style(**{"styleProperty": "style"}),
    {},
    "Businesses (red = flood vulnerable, green = not)",
)

m3.add_colorbar(
    vis_params={"min": 0, "max": 30, "palette": hand_palette},
    label="HAND (metres above nearest drainage)",
)

m3

Map(center=[np.float64(-6.216215325), np.float64(106.8375728)], controls=(WidgetControl(options=['position', '…

---
## Indicator 4: Tree Canopy Cover (ESA WorldCover 10m)

Shows ESA WorldCover land cover classification with 50m and 150m buffer circles around each business. Points are coloured by extracted canopy fraction (150m).

In [ ]:
# ============================================================
# INDICATOR 4: Canopy — ESA WorldCover 10m with 50m + 150m buffers
# ============================================================

worldcover = ee.ImageCollection("ESA/WorldCover/v200").mosaic().select("Map")
tree_mask = worldcover.eq(10).selfMask()  # Tree cover class only

m4 = geemap.Map(center=[center_lat, center_lon], zoom=15)
m4.add_basemap("SATELLITE")

# Full WorldCover classification (semi-transparent)
wc_vis = {
    "min": 10,
    "max": 100,
    "palette": [
        "#006400",  # 10 - Tree cover
        "#ffbb22",  # 20 - Shrubland
        "#ffff4c",  # 30 - Grassland
        "#f096ff",  # 40 - Cropland
        "#fa0000",  # 50 - Built-up
        "#b4b4b4",  # 60 - Bare / sparse veg
        "#f0f0f0",  # 70 - Snow and ice
        "#0064c8",  # 80 - Permanent water
        "#0096a0",  # 90 - Herbaceous wetland
        "#00cf75",  # 95 - Mangroves
        "#fae6a0",  # 100 - Moss and lichen
    ],
    "opacity": 0.5,
}
m4.addLayer(worldcover, wc_vis, "ESA WorldCover 2021 (all classes)", shown=False)

# Tree cover only — bright green
m4.addLayer(
    tree_mask,
    {"palette": ["#00FF00"], "opacity": 0.6},
    "Tree cover pixels (class 10)",
)

# 150m buffer rings (white)
buffers_150 = fc.map(lambda f: f.setGeometry(f.geometry().buffer(150)))
m4.addLayer(
    buffers_150.style(**{"color": "#FFFFFF", "fillColor": "#FFFFFF22", "width": 1.5}),
    {},
    "150m buffers",
)

# 50m buffer rings (yellow)
buffers_50 = fc.map(lambda f: f.setGeometry(f.geometry().buffer(50)))
m4.addLayer(
    buffers_50.style(**{"color": "#FFD700", "fillColor": "#FFD70022", "width": 1.5}),
    {},
    "50m buffers",
)

# Points coloured by canopy fraction (150m): white (0) → dark green (high canopy)
def style_canopy(f):
    frac = ee.Number(f.get("canopy_fraction_150m"))
    g_int = ee.Number(100).add(frac.multiply(155)).toInt().min(255)
    r_int = ee.Number(1).subtract(frac).multiply(255).toInt().max(0)
    color = ee.String("#").cat(
        r_int.format("%02x")
    ).cat(
        g_int.format("%02x")
    ).cat("00")
    return f.set("style", {
        "color": "#000000",
        "pointSize": 7,
        "fillColor": color,
        "width": 2,
    })

m4.addLayer(
    fc.map(style_canopy).style(**{"styleProperty": "style"}),
    {},
    "Businesses (colour = canopy fraction 150m)",
)

m4

---
## Indicator 5: Heavy Rainfall Days (CHIRPS Daily)

Shows total precipitation over the 2-year trailing window. Business points are coloured by number of heavy rain days (>20mm).

In [ ]:
# ============================================================
# INDICATOR 5: Rainfall — CHIRPS Daily total precipitation
# ============================================================

# Window used by the extraction, read from the data (not recomputed)
start_date, end_date = RAIN_START, RAIN_END

chirps = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterDate(start_date, end_date)
    .select("precipitation")
)

rain_total = chirps.sum()
rain_max = chirps.max()

rain_palette = ["#ffffcc", "#a1dab4", "#41b6c4", "#2c7fb8", "#253494"]

m5 = geemap.Map(center=[center_lat, center_lon], zoom=13)
m5.add_basemap("CartoDB.Positron")

m5.addLayer(
    rain_total,
    {"min": 500, "max": 4000, "palette": rain_palette},
    f"Total Precipitation (mm) [{start_date} to {end_date}]",
)

m5.addLayer(
    rain_max,
    {"min": 20, "max": 150, "palette": rain_palette},
    f"Max Single-Day Rainfall (mm) [{start_date} to {end_date}]",
    shown=False,
)

# Colour points by rain_days_gt20mm
rain_max_days = int(max(df["rain_days_gt20mm"].max(), 1))

def style_rain(f):
    days = ee.Number(f.get("rain_days_gt20mm"))
    ratio = days.divide(rain_max_days).min(1)
    # Light blue → dark blue
    r = ee.Number(1).subtract(ratio).multiply(200).toInt().max(0)
    g = ee.Number(1).subtract(ratio).multiply(200).toInt().add(55).min(255)
    b = ee.Number(148).add(ratio.multiply(107)).toInt().min(255)
    color = ee.String("#").cat(r.format("%02x")).cat(g.format("%02x")).cat(b.format("%02x"))
    return f.set("style", {
        "color": "#000000",
        "pointSize": 7,
        "fillColor": color,
        "width": 1.5,
    })

m5.addLayer(
    fc.map(style_rain).style(**{"styleProperty": "style"}),
    {},
    f"Businesses (colour = days > 20mm, max={rain_max_days})",
)

m5.add_colorbar(
    vis_params={"min": 500, "max": 4000, "palette": rain_palette},
    label="Total Precipitation (mm)",
)

m5

---
## Indicator 6: Air Quality — Aerosol Optical Depth (MODIS MAIAC)

Shows mean AOD over the 2-year trailing window. AOD is a satellite-derived proxy for PM2.5 particulate pollution. Higher values indicate worse air quality.

In [ ]:
# ============================================================
# INDICATOR 6: Air Quality — MODIS MAIAC AOD
# ============================================================

# Window used by the extraction, read from the data (not recomputed)
start_date, end_date = AOD_START, AOD_END

maiac = (
    ee.ImageCollection("MODIS/061/MCD19A2_GRANULES")
    .filterDate(start_date, end_date)
    .select("Optical_Depth_047")
)

# Convert to actual AOD (scale factor 0.001)
aod_mean = maiac.map(lambda img: img.multiply(0.001)).mean()
aod_max = maiac.map(lambda img: img.multiply(0.001)).max()

aod_palette = ["#1a9850", "#91cf60", "#d9ef8b", "#fee08b", "#fc8d59", "#d73027"]

m6 = geemap.Map(center=[center_lat, center_lon], zoom=13)
m6.add_basemap("CartoDB.DarkMatter")

m6.addLayer(
    aod_mean,
    {"min": 0, "max": 1.0, "palette": aod_palette},
    f"Mean AOD [{start_date} to {end_date}]",
)

m6.addLayer(
    aod_max,
    {"min": 0, "max": 3.0, "palette": aod_palette},
    f"Max AOD [{start_date} to {end_date}]",
    shown=False,
)

# Points coloured by mean AOD: green (clean) → red (polluted)
def style_aod(f):
    aod = ee.Number(f.get("aod_mean"))
    ratio = aod.divide(1.0).min(1).max(0)
    r = ratio.multiply(215).add(26).toInt().min(255)
    g = ee.Number(1).subtract(ratio).multiply(207).add(48).toInt().min(255)
    b = ee.Number(80).subtract(ratio.multiply(50)).toInt().max(0)
    color = ee.String("#").cat(r.format("%02x")).cat(g.format("%02x")).cat(b.format("%02x"))
    return f.set("style", {
        "color": "#FFFFFF",
        "pointSize": 7,
        "fillColor": color,
        "width": 1,
    })

m6.addLayer(
    fc.map(style_aod).style(**{"styleProperty": "style"}),
    {},
    "Businesses (colour = mean AOD)",
)

m6.add_colorbar(
    vis_params={"min": 0, "max": 1.0, "palette": aod_palette},
    label="Mean Aerosol Optical Depth (higher = worse air quality)",
)

m6

---
## Indicator 7: Nighttime Lights (VIIRS Monthly)

Shows mean nighttime radiance over the trailing 12 months. Higher radiance indicates greater economic activity and urbanisation intensity.

In [ ]:
# ============================================================
# INDICATOR 7: Nighttime Lights — VIIRS monthly composites
# ============================================================

# Window used by the extraction, read from the data (not recomputed)
start_date, end_date = NTL_START, NTL_END

viirs = (
    ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
    .filterDate(start_date, end_date)
    .select("avg_rad")
)

ntl_mean = viirs.mean()

ntl_palette = ["#000000", "#0d0887", "#7e03a8", "#cc4778", "#f89540", "#f0f921"]

m7 = geemap.Map(center=[center_lat, center_lon], zoom=13)
m7.add_basemap("CartoDB.DarkMatter")

m7.addLayer(
    ntl_mean,
    {"min": 0, "max": 60, "palette": ntl_palette},
    f"Mean Nighttime Radiance [{start_date} to {end_date}]",
)

# 150m buffer rings
buffers_150 = fc.map(lambda f: f.setGeometry(f.geometry().buffer(150)))
m7.addLayer(
    buffers_150.style(**{"color": "#FFFFFF44", "fillColor": "#FFFFFF11", "width": 1}),
    {},
    "150m buffers",
)

# Points coloured by radiance: dark → bright
ntl_max_val = int(max(df["ntl_mean_radiance"].max(), 1))

def style_ntl(f):
    rad = ee.Number(f.get("ntl_mean_radiance"))
    ratio = rad.divide(ntl_max_val).min(1).max(0)
    # Dark purple → yellow
    r = ratio.multiply(240).toInt().min(255)
    g = ratio.multiply(249).toInt().min(255)
    b = ee.Number(1).subtract(ratio).multiply(135).add(33).toInt().min(255)
    color = ee.String("#").cat(r.format("%02x")).cat(g.format("%02x")).cat(b.format("%02x"))
    return f.set("style", {
        "color": "#FFFFFF",
        "pointSize": 7,
        "fillColor": color,
        "width": 1,
    })

m7.addLayer(
    fc.map(style_ntl).style(**{"styleProperty": "style"}),
    {},
    f"Businesses (colour = mean radiance, max={ntl_max_val})",
)

m7.add_colorbar(
    vis_params={"min": 0, "max": 60, "palette": ntl_palette},
    label="Mean Nighttime Radiance (nW/cm²/sr)",
)

m7

---
## Indicator 8: Built-up Surface Fraction (JRC GHSL 10m)

Shows built-up surface density from the Global Human Settlement Layer. Points are coloured by built-up fraction within 150m buffer.

In [ ]:
# ============================================================
# INDICATOR 8: Built-up Surface — JRC GHSL 10m
# ============================================================

ghsl = ee.Image("JRC/GHSL/P2023A/GHS_BUILT_S/2020").select("built_surface")

# GHSL is 0-100 (percentage); normalise to 0-1 for display
ghsl_frac = ghsl.divide(100)

bu_palette = ["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"]

m8 = geemap.Map(center=[center_lat, center_lon], zoom=15)
m8.add_basemap("SATELLITE")

m8.addLayer(
    ghsl_frac,
    {"min": 0, "max": 1, "palette": bu_palette, "opacity": 0.6},
    "Built-up Surface Fraction (GHSL 2020)",
)

# 150m buffer rings
buffers_150 = fc.map(lambda f: f.setGeometry(f.geometry().buffer(150)))
m8.addLayer(
    buffers_150.style(**{"color": "#FFFFFF", "fillColor": "#FFFFFF22", "width": 1.5}),
    {},
    "150m buffers",
)

# 50m buffer rings
buffers_50 = fc.map(lambda f: f.setGeometry(f.geometry().buffer(50)))
m8.addLayer(
    buffers_50.style(**{"color": "#00FFFF", "fillColor": "#00FFFF22", "width": 1.5}),
    {},
    "50m buffers",
)

# Points coloured by built-up fraction (150m): yellow (low) → dark red (high)
def style_builtup(f):
    frac = ee.Number(f.get("builtup_fraction_150m"))
    ratio = frac.min(1).max(0)
    r = ee.Number(189).add(ratio.multiply(66)).toInt().min(255)
    g = ee.Number(255).subtract(ratio.multiply(217)).toInt().max(0)
    b = ee.Number(178).subtract(ratio.multiply(140)).toInt().max(0)
    color = ee.String("#").cat(r.format("%02x")).cat(g.format("%02x")).cat(b.format("%02x"))
    return f.set("style", {
        "color": "#000000",
        "pointSize": 7,
        "fillColor": color,
        "width": 2,
    })

m8.addLayer(
    fc.map(style_builtup).style(**{"styleProperty": "style"}),
    {},
    "Businesses (colour = built-up fraction 150m)",
)

m8.add_colorbar(
    vis_params={"min": 0, "max": 1, "palette": bu_palette},
    label="Built-up Surface Fraction (0 = none, 1 = fully built)",
)

m8